# 03 — Evolutionary scores

Default: load the precomputed CSV (ESM2 35M/650M/3B, EVmutation2, plus EVE / EVmutation v1 columns when present). Optional GPU re-run uses published `evedesign` and a **user-supplied** EVmutation2 checkpoint.


In [ ]:
from pathlib import Path

import pandas as pd

from dualde.paths import example_dir

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

evo = pd.read_csv(EXAMPLE / "scores" / "evolutionary.csv")
print(evo.shape)
print(evo.columns.tolist())
print(evo.head())
evo.to_csv(OUT / "evolutionary.csv", index=False)


## Optional: re-run ESM2 + EVmutation2 via PyPI evedesign

Requires `pip install -e ".[evo]"`, a GPU is recommended, and `EVMUTATION2_CKPT` pointing at a local `.ckpt` (not shipped). Also set `HF_HOME` / `TORCH_HOME`.


In [ ]:
RUN_EVEDESIGN = False  # flip to True to re-score
EVMUTATION2_CKPT = None  # e.g. Path(".../epoch=12-step=95987.ckpt")
DEVICE = "cuda"  # or "cpu"

if RUN_EVEDESIGN:
    import os
    from dualde.mapping import read_fasta

    from evedesign.models import esm2, evmutation2
    from evedesign.sequence import Sequences
    from evedesign.system import EntityInstance, Protein, System, SystemInstance

    header, sequence = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
    a3m = EXAMPLE / "aln" / "NUDT15_HUMAN_b0.7.a3m"
    alignment = Sequences.from_file(str(a3m)) if a3m.is_file() else None
    protein_id = "NUDT15_HUMAN"
    target_inst = SystemInstance(EntityInstance(rep=sequence))

    esm2_models = [
        ("esm2_t12_35M_UR50D", "ESM2_35M"),
        ("esm2_t33_650M_UR50D", "ESM2_650M"),
        ("esm2_t36_3B_UR50D", "ESM2_3B"),
    ]
    results = []
    for model_name, score_col_name in esm2_models:
        print(score_col_name, model_name)
        esm = esm2.ESM2(
            model_name=model_name,
            model_dir_path=None,
            batch_size=64,
            keep_model_after_build=True,
            device=DEVICE,
            num_sweeps=1000,
            init_strategy="system",
            scan_order="random",
        )
        system = System(Protein(id=protein_id, rep=sequence, first_index=1, sequences=alignment))
        esm.build(system)
        scan = esm.single_mutation_scan(target_inst)
        scan_df = (
            scan.stack().to_frame("score").reset_index()
            .rename(columns={"level_3": "aa_to"})
            .assign(mutation=lambda df: df["ref"] + df["pos"].astype(str) + df["aa_to"])
            [["pos", "ref", "aa_to", "mutation", "score"]]
            .rename(columns={"score": score_col_name})
        )
        results.append(scan_df)

    merged = results[0]
    for df in results[1:]:
        merged = merged.merge(df, on=["pos", "ref", "aa_to", "mutation"], how="outer")

    if alignment is not None and EVMUTATION2_CKPT:
        evm = evmutation2.EVmutation2(
            model_file_path=str(EVMUTATION2_CKPT),
            encoder_num_samples=4,
            encoder_num_recycling_steps=4,
            encoder_max_num_msa=1024,
            decoder_batch_size=64,
            decoder_num_full_samples=16,
            decoder_num_mutant_samples=1,
            keep_model_after_build=True,
            device=DEVICE,
        )
        system = System(Protein(id=protein_id, rep=sequence, first_index=1, sequences=alignment))
        evm.build(system)
        scan = evm.single_mutation_scan(target_inst)
        evm_df = (
            scan.stack().to_frame("score").reset_index()
            .rename(columns={"level_3": "aa_to"})
            .assign(mutation=lambda df: df["ref"] + df["pos"].astype(str) + df["aa_to"])
            [["pos", "ref", "aa_to", "mutation", "score"]]
            .rename(columns={"score": "EVmutation2"})
        )
        merged = merged.merge(evm_df, on=["pos", "ref", "aa_to", "mutation"], how="outer")

    merged = merged.sort_values("pos").reset_index(drop=True)
    merged.to_csv(OUT / "evolutionary.csv", index=False)
    print(merged.head())
else:
    print("Skipping evedesign re-run (RUN_EVEDESIGN=False); using scores/evolutionary.csv")
